In [12]:
import pandas as pd
import torch

In [13]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [14]:
df = pd.read_csv('data/Eng-Fra.csv')
df.head()

,English,French
0,Go.,Va !
1,Go.,Marche.
2,Go.,En route !
3,Go.,Bouge !
4,Hi.,Salut !


In [15]:
from preprocessing.normalize import normalize_text

df['English'] = df['English'].apply(normalize_text)
df['French'] = df['French'].apply(normalize_text)

In [16]:
df.head()

,English,French
0,go,va
1,go,marche
2,go,en route
3,go,bouge
4,hi,salut


In [17]:
pairs = []

for eng, fr in zip(df['English'], df['French']):
    pairs.append((eng,fr))

In [18]:
len(pairs)

239189

In [19]:
from preprocessing.vocabulary import Vocabulary

eng_vocab = Vocabulary("eng")
fr_vocab = Vocabulary("fr")

# Step 1: count words
for eng, fr in pairs:
    eng_vocab.add_sentence(eng)
    fr_vocab.add_sentence(fr)

# Step 2: build vocab (top-K)
eng_vocab.build_vocab(max_size=10000)
fr_vocab.build_vocab(max_size=10000)

In [20]:
from preprocessing.sentence_to_tensor import sentence_to_tensor
print(eng_vocab.vocab_size)
print(fr_vocab.vocab_size)

print(sentence_to_tensor(eng_vocab, "i am cold"))

10004
10004
tensor([  0,   4, 147, 306,   1])


In [21]:
from torch.utils.data import DataLoader
from preprocessing.dataset import TranslationDataset, collate_fn

dataset = TranslationDataset(pairs, eng_vocab, fr_vocab)

dataloader = DataLoader(
    dataset,
    batch_size = 32,
    shuffle = True,
    collate_fn = collate_fn
)

In [22]:
for src, trg in dataloader:
    print(src.shape)
    print(trg.shape)
    break

torch.Size([32, 12])
torch.Size([32, 15])


In [23]:
from models.seq2seq import Seq2Seq
from models.encoder import Encoder
from models.decoder import Decoder

encoder = Encoder(
    vocab_size = eng_vocab.vocab_size, 
    embedding_dim=256, 
    hidden_size = 512
    )
decoder = Decoder(
    vocab_size = fr_vocab.vocab_size, 
    embedding_dim= 256, 
    hidden_size = 512
    )

model = Seq2Seq(encoder, decoder, device)

In [24]:
print(encoder)
print(decoder)

Encoder(
  (embedding): Embedding(10004, 256)
  (gru): GRU(256, 512, batch_first=True)
)
Decoder(
  (embedding): Embedding(10004, 256)
  (gru): GRU(768, 512, batch_first=True)
  (fc): Linear(in_features=512, out_features=10004, bias=True)
  (attention): Attention(
    (attn): Linear(in_features=1024, out_features=512, bias=True)
    (v): Linear(in_features=512, out_features=1, bias=False)
  )
)


In [25]:
PAD_IDX = 2 # the tokens which should be ignored and must not affect the training process

criterion = torch.nn.CrossEntropyLoss(ignore_index = PAD_IDX)
optimizer = torch.optim.Adam(model.parameters(), lr = 0.001)

In [ ]:
num_epochs = 10
for epoch in range(num_epochs):
    total_loss = 0
    for src, trg in dataloader:

        src = src.to(device)
        trg = trg.to(device)

        optimizer.zero_grad()

        decoder_input = trg[:, :-1]
        target = trg[:, 1:]

        output = model(src, decoder_input)

        output = output.reshape(-1, output.shape[-1])
        target = target.reshape(-1)

        loss = criterion(output, target)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    
    avg_loss = total_loss/ len(dataloader)
    print(f"Epoch: {epoch+1}, Loss: {avg_loss:.4f}")

KeyboardInterrupt: 